In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
from datetime import time
warnings.filterwarnings("ignore")

In [ ]:
# Generate ID
def combine_id(row):
    return f"{row['contract']}_{row['strike_price']}_{row['cp']}"

def find_closest_data(strike_price, row, filtered_data, max_minutes=5):
    for i in range(1, max_minutes + 1):
        timestamp = row["timestamp"] - pd.Timedelta(minutes=i)
        timestamp_data = filtered_data[(filtered_data["timestamp"] == timestamp) & (filtered_data["strike_price"] == strike_price)]
        
        if len(timestamp_data) == 2:
            call = timestamp_data[timestamp_data["cp"] == "C"]["c"]
            put = timestamp_data[timestamp_data["cp"] == "P"]["c"]
            return float(call), float(put)
            
    return np.nan, np.nan

def calculate_ATMS(row, option):
    timestamp_data = option[option["timestamp"] == row["timestamp"]]
    High_1_SP = row["High_1_SP"]
    Low_1_SP = row["Low_1_SP"]

    def safe_get_value(data, condition):
        result = data[condition]["c"]
        return float(result) if not result.empty else np.nan

    ITM_1_call = safe_get_value(timestamp_data, (timestamp_data["strike_price"] == Low_1_SP) & (timestamp_data["cp"] == "C"))
    ITM_1_put = safe_get_value(timestamp_data, (timestamp_data["strike_price"] == High_1_SP) & (timestamp_data["cp"] == "P"))
    OTM_1_call = safe_get_value(timestamp_data, (timestamp_data["strike_price"] == High_1_SP) & (timestamp_data["cp"] == "C"))
    OTM_1_put = safe_get_value(timestamp_data, (timestamp_data["strike_price"] == Low_1_SP) & (timestamp_data["cp"] == "P"))

    ITM_1_call_TV = np.nan
    ITM_1_put_TV = np.nan
    Low_1_sum = np.nan
    High_1_sum = np.nan
    ATMS = np.nan

    if not (np.isnan(ITM_1_call) or np.isnan(ITM_1_put) or np.isnan(OTM_1_call) or np.isnan(OTM_1_put)):
        ITM_1_call_TV = ITM_1_call - (row['TX'] - Low_1_SP)
        ITM_1_put_TV = ITM_1_put - (High_1_SP - row['TX'])

        Low_1_sum = ITM_1_call_TV + OTM_1_put
        High_1_sum = OTM_1_call + ITM_1_put_TV

        if High_1_SP == Low_1_SP:
            ATMS = ((Low_1_sum + High_1_sum) / 2) / row['TX'] * 100
        else:
            ATMS = ((Low_1_sum * (High_1_SP - row['TX']) / (High_1_SP - Low_1_SP)) + 
                    (High_1_sum * (row['TX'] - Low_1_SP) / (High_1_SP - Low_1_SP))) / row['TX'] * 100

    return pd.Series([ITM_1_call, ITM_1_call_TV, ITM_1_put, ITM_1_put_TV, OTM_1_call, OTM_1_put, ATMS],
                     index=["ITM_1_call", "ITM_1_call_TV", "ITM_1_put", "ITM_1_put_TV", "OTM_1_call", "OTM_1_put", "ATMS"])

def find_SP(row):
    result = {}
    for i in range(1, 6):
        result[f"Low_{i}_SP"] = int(((row['TX'] // 50) * 50) - (50 * (i - 1)))
        if row['TX'] % 50 == 0:
            result[f"High_{i}_SP"] = int((((row['TX'] // 50) // 50) * 50) + (50 * (i - 1)))
        else:
            result[f"High_{i}_SP"] = int(((row['TX'] // 50) * 50) + (50 * i))
    return pd.Series(result)

In [ ]:
TX_data = pd.read_csv("data/TX_data.csv")

In [ ]:
folder_path = "data\option"
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

In [ ]:
def is_trading_time(ts):
    #Determines whether the given timestamp falls within the trading hours.

    w = ts.weekday()
    t = ts.time()

    def in_range(t, start, end):
        return (start <= t < end)

    # Wednesday night session (Wed 15:00 - Thu 05:00)
    if (w == 2 and t >= time(15, 0)) or (w == 3 and t < time(5, 0)):
        return True

    # Thursday day session (Thu 08:45 - 13:45)
    if w == 3 and in_range(t, time(8, 45), time(13, 45)):
        return True

    # Thursday night session (Thu 15:00 - Fri 05:00)
    if (w == 3 and t >= time(15, 0)) or (w == 4 and t < time(5, 0)):
        return True

    # Friday day session (Fri 08:45 - 13:45)
    if w == 4 and in_range(t, time(8, 45), time(13, 45)):
        return True

    # Friday night session (Fri 15:00 - Sat 05:00)
    if (w == 4 and t >= time(15, 0)) or (w == 5 and t < time(5, 0)):
        return True

    # Monday day session (Mon 08:45 - 13:45)
    if w == 0 and in_range(t, time(8, 45), time(13, 45)):
        return True

    # Monday night session (Mon 15:00 - Tue 05:00)
    if (w == 0 and t >= time(15, 0)) or (w == 1 and t < time(5, 0)):
        return True

    # Tuesday day session (Tue 08:45 - 13:45)
    if w == 1 and in_range(t, time(8, 45), time(13, 45)):
        return True

    # Tuesday night session (Tue 15:00 - Wed 05:00)
    if (w == 1 and t >= time(15, 0)) or (w == 2 and t < time(5, 0)):
        return True

    # Wednesday day session (Wed 08:45 - 13:30)
    if w == 2 and in_range(t, time(8, 45), time(13, 30)):
        return True

    # All other times are non-trading hours
    return False

In [ ]:
for option_file in csv_files[282:]:

    file_path = os.path.join(folder_path, option_file)

    option = pd.read_csv(file_path)

    # Generate timestamp
    option['timestamp'] = pd.to_datetime(option['date'] + ' ' + option['time'])

    # Drop data before 2019-01-02 08:45:00 and after 2024-06-19 13:30:00
    start_date = pd.to_datetime('2019-01-02 08:45:00')
    end_date = pd.to_datetime('2025-01-01 00:00:00')
    option = option[(option['timestamp'] >= start_date) & (option['timestamp'] < end_date)]

    # Load / Process TX_data
    TX_data['timestamp'] = pd.to_datetime(TX_data['timestamp'])

    # Merge
    time_data = pd.merge(
        TX_data[['timestamp', "T_open", 'T_high', 'T_low', 'T_close',
                 'F_open', 'F_high', 'F_low', 'F_close', 'F_vol', 'TX']],
        option[['timestamp']],
        on='timestamp',
        how='inner'
    ).drop_duplicates(subset='timestamp', keep='first')

    # Calculate required columns
    time_data[["Low_1_SP","High_1_SP","Low_2_SP","High_2_SP","Low_3_SP","High_3_SP",
               "Low_4_SP","High_4_SP","Low_5_SP","High_5_SP"]] = time_data.apply(find_SP, axis=1)
    
    data_list = ["ITM_1_call","ITM_1_call_TV","ITM_1_put","ITM_1_put_TV","OTM_1_call","OTM_1_put","ATMS"]

    time_data[data_list] = time_data.apply(calculate_ATMS, args=(option,), axis=1)

    time_data = time_data.sort_values(by='timestamp')

    time_data.set_index('timestamp', inplace=True)

    min_ts = time_data.index[0]
    max_ts = time_data.index[-1]

    full_range = pd.date_range(start=min_ts, end=max_ts, freq='T')

    valid_minutes = [ts for ts in full_range if is_trading_time(ts)]

    time_data = time_data.reindex(valid_minutes)

    last_idx = time_data.index[-1]
    time_data.loc[last_idx, 'ATMS'] = 0

    time_data['ATMS_interpolated'] = time_data['ATMS'].interpolate(
        method='linear', limit_direction='both'
    )

    time_data['TX'] = time_data['TX'].interpolate(
        method='linear', limit_direction='both'
    )

    # Specify columns to fill
    columns_to_fill = ["T_open", "T_high", "T_low", "T_close", "F_open", "F_high", "F_low", "F_close","F_vol",
                        "Low_1_SP", "High_1_SP", "Low_2_SP", "High_2_SP", "Low_3_SP", "High_3_SP", "Low_4_SP", "High_4_SP", "Low_5_SP", "High_5_SP"]

    # Fill using forward fill
    time_data[columns_to_fill] = time_data[columns_to_fill].fillna(method='ffill')
    time_data.reset_index(inplace=True)
    time_data.rename(columns={'index':'timestamp'}, inplace=True)

    # Calculate remaining time
    last_timestamp = time_data['timestamp'].iloc[-1]
    time_data['remaining_time'] = ((last_timestamp - time_data['timestamp']).dt.total_seconds() / 60)+1

    # Generate time_label
    time_data['time_label'] = time_data['timestamp'].apply(lambda x: f"week {x.weekday() + 1} {x.strftime('%H:%M')}")

    # Output CSV
    out_path = f"data/ATMS_data/{option_file}"
    time_data.to_csv(out_path, encoding="utf_8_sig", index=False)

    print(f"{option_file} OK!!")